<a href="https://colab.research.google.com/github/SRIJANRAOS/srijanraos_INFO5731_spring2026/blob/main/In_Class_Exercise_5%266_Feature_Extraction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving product_reviews.txt to product_reviews.txt
Uploaded file name: product_reviews.txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [5]:
import pandas as pd

# Use correct filename
filename = "product_reviews.txt"

# Read file (tab-separated, with header)
df = pd.read_csv(filename, sep="\t")

# Check columns
print("Columns:", df.columns)

# (a) Shape
print("Shape:", df.shape)

# (b) First 3 rows
print("\nFirst 3 rows:")
print(df.head(3))

Columns: Index(['id', 'text'], dtype='object')
Shape: (10, 2)

First 3 rows:
   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [7]:
# Write your Answer here (Q2)
import re
import pandas as pd


In [8]:
def clean_text(text):
    text = str(text).lower()  # lowercase

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove emojis & non-ascii
    text = re.sub(r"[^\x00-\x7F]+", "", text)

    # remove special characters (keep letters & numbers)
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [9]:
df["clean_text"] = df["text"].apply(clean_text)

In [10]:
print(df[["text", "clean_text"]].head(5))

                                                text  \
0  Love this blender! Smoothies are super creamy ...   
1  Terrible quality... stopped working after 2 da...   
2      Good value for the price. Shipping was quick.   
3  Not as described. Missing parts and the box wa...   
4  Customer support helped me get a replacement. ...   

                                          clean_text  
0  love this blender smoothies are super creamy a...  
1      terrible quality stopped working after 2 days  
2        good value for the price shipping was quick  
3  not as described missing parts and the box was...  
4  customer support helped me get a replacement t...  


In [11]:
before_rows = df.shape[0]

# Remove empty rows after cleaning
df = df[df["clean_text"].str.strip() != ""]

after_rows = df.shape[0]

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", after_rows)

Rows before cleaning: 10
Rows after cleaning: 10


In [12]:
df.to_csv("cleaned_reviews.csv", index=False)

## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [13]:
# # Write your Answer here (Q3)

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Create CountVectorizer with English stopwords
vectorizer = CountVectorizer(stop_words="english")

# Fit and transform the text column
X = vectorizer.fit_transform(df["text"])

# --- Vocabulary size ---
vocab_size = len(vectorizer.get_feature_names_out())
print("Vocabulary size:", vocab_size)

# --- Top 10 words by total count ---
word_counts = np.array(X.sum(axis=0)).flatten()
words = vectorizer.get_feature_names_out()

# Pair words with counts
word_freq = list(zip(words, word_counts))

# Sort by count (descending)
word_freq_sorted = sorted(word_freq, key=lambda x: x[1], reverse=True)

print("\nTop 10 words by total count:")
for word, count in word_freq_sorted[:10]:
    print(word, ":", count)

Vocabulary size: 50

Top 10 words by total count:
amazing : 1
app : 1
battery : 1
blender : 1
box : 1
buy : 1
charged : 1
clear : 1
crashing : 1
creamy : 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [14]:
# Write your Answer here (Q4)
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Create CountVectorizer for BIGRAMS only
bigram_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 2)
)

# Fit and transform
X2 = bigram_vectorizer.fit_transform(df["text"])

# Get bigram counts across all documents
bigram_counts = np.array(X2.sum(axis=0)).flatten()
bigrams = bigram_vectorizer.get_feature_names_out()

# Pair bigrams with counts
bigram_freq = list(zip(bigrams, bigram_counts))

# Sort by count (descending)
bigram_freq_sorted = sorted(bigram_freq, key=lambda x: x[1], reverse=True)

# Print top 5 bigrams
print("Top 5 bigrams by total count:")
for bg, count in bigram_freq_sorted[:5]:
    print(bg, ":", count)

Top 5 bigrams by total count:
amazing screen : 1
app keeps : 1
battery life : 1
blender smoothies : 1
box damaged : 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [15]:
# Write your Answer here  (Q5)

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Create CountVectorizer for BIGRAMS only
bigram_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 2)
)

# Fit and transform
X2 = bigram_vectorizer.fit_transform(df["text"])

# Get bigram counts across all documents
bigram_counts = np.array(X2.sum(axis=0)).flatten()
bigrams = bigram_vectorizer.get_feature_names_out()

# Pair bigrams with counts
bigram_freq = list(zip(bigrams, bigram_counts))

# Sort by count (descending)
bigram_freq_sorted = sorted(bigram_freq, key=lambda x: x[1], reverse=True)

# Print top 5 bigrams
print("Top 5 bigrams by total count:")
for bg, count in bigram_freq_sorted[:5]:
    print(bg, ":", count)

Top 5 bigrams by total count:
amazing screen : 1
app keeps : 1
battery life : 1
blender smoothies : 1
box damaged : 1


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
